# Comparación de Estrategias Federadas vs Proactive Forest Base

## Objetivo
Comparar las métricas (Accuracy y Macro-F1) de cada estrategia federada (S1-S7 + PW) con los resultados base de Proactive Forest reportados por Nayma, usando **3 clientes** y promediando las métricas de los 3 clientes.

## Datasets
Car, Iris, Letter, Nursery, Optdigits, Sonar, Spambase, Vowel

## Configuración fija
- **n_clients = 3**
- **seed = 42**
- **n_estimators = 100**
- **distribution = iid**
- **alpha_pf = 0.45**
- **t_max = 100** (S2-S7)
- **local_weight = 0.5**
- **f1_weight = 0.6** (S4, S7, PW)
- **window_size = 7** (PW)
- **max_rounds = 15** (PW)
- **convergence_threshold = 0.002** (PW)

---
## 0. Imports y configuración común

In [1]:
import sys
from pathlib import Path

# Notebook está en src/interfaces/notebooks/ → buscar project root
ROOT = Path.cwd().resolve()
while not (ROOT / 'src').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.infrastructure.dataset.dataset_factory import DatasetFactory

print(f'Project root: {ROOT}')
assert (ROOT / 'src').exists(), f'No se encontró src/ en {ROOT}'

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from src.application.orchestrators import FLEXOrchestrator
from src.domain.dataset.base_adapter import DatasetSplit

SEED = 42
N_CLIENTS = 3
N_ESTIMATORS = 100
ALPHA_PF = 0.45
T_MAX = 100
LOCAL_WEIGHT = 0.5
F1_WEIGHT = 0.6

np.random.seed(SEED)

Project root: C:\Users\Adrián Rodríguez\Documents\! Study 📝\📊 KDD 🤖🧠\! Federated learning\federated_proactive_forest


---
## 1. Dataset Loaders

In [2]:
# ============================================================================
# DATASET LOADING CONFIGURATION
# ============================================================================
# Using DatasetFactory to standardize all dataset adapters.

def load_dataset(name):
    """Standardized loader using DatasetFactory."""
    return DatasetFactory.load_from_config({"type": name.capitalize()}, project_root=ROOT)

DATASETS = {
    'car': lambda: load_dataset('car'),
    'iris': lambda: load_dataset('iris'),
    'letter': lambda: load_dataset('letter'),
    'nursery': lambda: load_dataset('nursery'),
    'optdigits': lambda: load_dataset('optdigits'),
    'sonar': lambda: load_dataset('sonar'),
    'spambase': lambda: load_dataset('spambase'),
    'vowel': lambda: load_dataset('vowel'),
}


---
## 2. Función de experimento genérica

In [3]:
def build_config(strategy, n_clients=N_CLIENTS, n_estimators=N_ESTIMATORS, seed=SEED):
    """Construye config con los parámetros unificados para cualquier estrategia."""
    cfg = {
        'federation': {'n_clients': n_clients, 'distribution': 'iid', 'seed': seed},
        'model': {
            'n_estimators': n_estimators, 'alpha': ALPHA_PF,
            'split_criterion': 'entropy', 'use_progressive_stopping': True,
            'convergence': 0.002, 'episode_size': 5, 'verbose': False,
        },
        'aggregation': {'strategy': strategy, 't_max': T_MAX},
        'prediction': {'local_weight': LOCAL_WEIGHT, 'global_weight': 1.0 - LOCAL_WEIGHT},
        'verbose': False, 'seed': seed,
    }

    # S4, S7, PW → f1_weight
    if strategy in ('S4', 'S7', 'PW'):
        cfg['aggregation']['f1_weight'] = F1_WEIGHT
        cfg['aggregation']['pcd_weight'] = 1.0 - F1_WEIGHT

    # PW → parámetros exclusivos
    if strategy == 'PW':
        cfg['aggregation']['window_size'] = 7
        cfg['aggregation']['max_rounds'] = 15
        cfg['aggregation']['convergence_threshold'] = 0.002

    return cfg


def run_experiment(dataset_name, strategy):
    """Ejecuta una combinación dataset × estrategia y devuelve dict con métricas por cliente."""
    ds = DATASETS[dataset_name]()
    cfg = build_config(strategy)
    np.random.seed(SEED)

    orch = FLEXOrchestrator.from_config(cfg)
    orch.setup_federation(ds, seed=SEED)
    results = orch.run_federated_round()

    # Métricas por cliente (híbridas)
    client_metrics = {}
    for cid, preds in results.client_hybrid_predictions.items():
        acc = accuracy_score(results.y_test, preds)
        f1 = f1_score(results.y_test, preds, average='macro', zero_division=0)
        client_metrics[cid] = {'accuracy': acc, 'macro_f1': f1}

    return {
        'dataset': dataset_name,
        'strategy': strategy,
        'global_accuracy': results.global_accuracy,
        'global_macro_f1': results.global_macro_f1,
        'client_metrics': client_metrics,
        'n_trees_global': results.n_trees_global,
    }

---
## 3. Ejecutar cada estrategia (celda independiente por si acaso)

> **Nota:** Cada celda ejecuta la estrategia para **todos los datasets**. Los resultados se acumulan en `all_results`.

In [4]:
# Celda compartida para almacenar resultados
all_results = {}  # {(dataset, strategy): result_dict}

In [5]:
# ── S1: Simple Pool ──────────────────────────────────────────────────────
STRATEGY = 'S1'
print(f"\n{'='*60}\n🚀 Ejecutando {STRATEGY}\n{'='*60}")
for ds_name in DATASETS:
    print(f"  ▶ {ds_name}...", end=' ', flush=True)
    res = run_experiment(ds_name, STRATEGY)
    all_results[(ds_name, STRATEGY)] = res
    print(f"F1={res['global_macro_f1']:.4f}")
print(f"✅ {STRATEGY} completada.")


🚀 Ejecutando S1
  ▶ car... 
🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 37 árboles entrenados
  client_1: 33 árboles entrenados
  client_2: 16 árboles entrenados
  TOTAL: 86 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S1)
  client_0: 37/37 árboles seleccionados
  client_1: 33/33 árboles seleccionados
  client_2: 16/16 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 86 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 37 locales + 49 globales externos (37 propios excluidos) = 86 árboles
  client_1: 33 locales + 53 globales externos (33 propios excluidos) = 86 árboles
  client_2: 16 locales + 70 globales externos (16 propios excluidos) = 86 árboles

F1=0.8623
  ▶ iris... 
🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 49 árboles entrenados
  client_1: 16 árboles entrenados
  client_2: 16 árboles entrenados
  TOTAL: 81 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S1)
  client_0: 49/49 árboles seleccionados
  client_1: 16/16 árboles seleccionados
  client_2: 16/16 

In [6]:
# ── S2: Global Accuracy + PF ─────────────────────────────────────────────
STRATEGY = 'S2'
print(f"\n{'='*60}\n🚀 Ejecutando {STRATEGY}\n{'='*60}")
for ds_name in DATASETS:
    print(f"  ▶ {ds_name}...", end=' ', flush=True)
    res = run_experiment(ds_name, STRATEGY)
    all_results[(ds_name, STRATEGY)] = res
    print(f"F1={res['global_macro_f1']:.4f}")
print(f"✅ {STRATEGY} completada.")


🚀 Ejecutando S2
  ▶ car... 
🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 63 árboles entrenados
  client_1: 90 árboles entrenados
  client_2: 16 árboles entrenados
  TOTAL: 169 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S2)
  client_0: 63/63 árboles seleccionados
  client_1: 90/90 árboles seleccionados
  client_2: 16/16 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 169 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 63 locales + 106 globales externos (63 propios excluidos) = 169 árboles
  client_1: 90 locales + 79 globales externos (90 propios excluidos) = 169 árboles
  client_2: 16 locales + 153 globales externos (16 propios excluidos) = 169 árboles

F1=0.9101
  ▶ iris... 
🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 9 árboles entrenados
  client_1: 16 árboles entrenados
  client_2: 16 árboles entrenados
  TOTAL: 41 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S2)
  client_0: 9/9 árboles seleccionados
  client_1: 16/16 árboles seleccionados
  client_2: 16

In [7]:
# ── S3: Global Macro-F1 + PF ─────────────────────────────────────────────
STRATEGY = 'S3'
print(f"\n{'='*60}\n🚀 Ejecutando {STRATEGY}\n{'='*60}")
for ds_name in DATASETS:
    print(f"  ▶ {ds_name}...", end=' ', flush=True)
    res = run_experiment(ds_name, STRATEGY)
    all_results[(ds_name, STRATEGY)] = res
    print(f"F1={res['global_macro_f1']:.4f}")
print(f"✅ {STRATEGY} completada.")


🚀 Ejecutando S3
  ▶ car... 
🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 56 árboles entrenados
  client_1: 27 árboles entrenados
  client_2: 16 árboles entrenados
  TOTAL: 99 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S3)
  client_0: 56/56 árboles seleccionados
  client_1: 27/27 árboles seleccionados
  client_2: 16/16 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 99 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 56 locales + 43 globales externos (56 propios excluidos) = 99 árboles
  client_1: 27 locales + 72 globales externos (27 propios excluidos) = 99 árboles
  client_2: 16 locales + 83 globales externos (16 propios excluidos) = 99 árboles

F1=0.8872
  ▶ iris... 
🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 27 árboles entrenados
  client_1: 35 árboles entrenados
  client_2: 27 árboles entrenados
  TOTAL: 89 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S3)
  client_0: 27/27 árboles seleccionados
  client_1: 35/35 árboles seleccionados
  client_2: 27/27 

In [8]:
# ── S4: Global F1 + PCD + PF ─────────────────────────────────────────────
STRATEGY = 'S4'
print(f"\n{'='*60}\n🚀 Ejecutando {STRATEGY}\n{'='*60}")
for ds_name in DATASETS:
    print(f"  ▶ {ds_name}...", end=' ', flush=True)
    res = run_experiment(ds_name, STRATEGY)
    all_results[(ds_name, STRATEGY)] = res
    print(f"F1={res['global_macro_f1']:.4f}")
print(f"✅ {STRATEGY} completada.")


🚀 Ejecutando S4
  ▶ car... 
🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 37 árboles entrenados
  client_1: 33 árboles entrenados
  client_2: 16 árboles entrenados
  TOTAL: 86 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S4)
  client_0: 37/37 árboles seleccionados
  client_1: 33/33 árboles seleccionados
  client_2: 16/16 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 86 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 37 locales + 49 globales externos (37 propios excluidos) = 86 árboles
  client_1: 33 locales + 53 globales externos (33 propios excluidos) = 86 árboles
  client_2: 16 locales + 70 globales externos (16 propios excluidos) = 86 árboles

F1=0.8426
  ▶ iris... 
🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 9 árboles entrenados
  client_1: 24 árboles entrenados
  client_2: 37 árboles entrenados
  TOTAL: 70 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S4)
  client_0: 9/9 árboles seleccionados
  client_1: 24/24 árboles seleccionados
  client_2: 37/37 árb

In [9]:
# ── S5: Per-Client Accuracy + PF ─────────────────────────────────────────
STRATEGY = 'S5'
print(f"\n{'='*60}\n🚀 Ejecutando {STRATEGY}\n{'='*60}")
for ds_name in DATASETS:
    print(f"  ▶ {ds_name}...", end=' ', flush=True)
    res = run_experiment(ds_name, STRATEGY)
    all_results[(ds_name, STRATEGY)] = res
    print(f"F1={res['global_macro_f1']:.4f}")
print(f"✅ {STRATEGY} completada.")


🚀 Ejecutando S5
  ▶ car... 
🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 27 árboles entrenados
  client_1: 27 árboles entrenados
  client_2: 24 árboles entrenados
  TOTAL: 78 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S5)
  client_0: 27/27 árboles seleccionados
  client_1: 27/27 árboles seleccionados
  client_2: 24/24 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 78 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 27 locales + 51 globales externos (27 propios excluidos) = 78 árboles
  client_1: 27 locales + 51 globales externos (27 propios excluidos) = 78 árboles
  client_2: 24 locales + 54 globales externos (24 propios excluidos) = 78 árboles

F1=0.8642
  ▶ iris... 
🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 18 árboles entrenados
  client_1: 24 árboles entrenados
  client_2: 24 árboles entrenados
  TOTAL: 66 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S5)
  client_0: 18/18 árboles seleccionados
  client_1: 24/24 árboles seleccionados
  client_2: 24/24 

In [10]:
# ── S6: Per-Client Macro-F1 + PF ─────────────────────────────────────────
STRATEGY = 'S6'
print(f"\n{'='*60}\n🚀 Ejecutando {STRATEGY}\n{'='*60}")
for ds_name in DATASETS:
    print(f"  ▶ {ds_name}...", end=' ', flush=True)
    res = run_experiment(ds_name, STRATEGY)
    all_results[(ds_name, STRATEGY)] = res
    print(f"F1={res['global_macro_f1']:.4f}")
print(f"✅ {STRATEGY} completada.")


🚀 Ejecutando S6
  ▶ car... 
🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 27 árboles entrenados
  client_1: 46 árboles entrenados
  client_2: 24 árboles entrenados
  TOTAL: 97 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S6)
  client_0: 27/27 árboles seleccionados
  client_1: 46/46 árboles seleccionados
  client_2: 24/24 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 97 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 27 locales + 70 globales externos (27 propios excluidos) = 97 árboles
  client_1: 46 locales + 51 globales externos (46 propios excluidos) = 97 árboles
  client_2: 24 locales + 73 globales externos (24 propios excluidos) = 97 árboles

F1=0.9037
  ▶ iris... 
🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 16 árboles entrenados
  client_1: 24 árboles entrenados
  client_2: 50 árboles entrenados
  TOTAL: 90 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S6)
  client_0: 16/16 árboles seleccionados
  client_1: 24/24 árboles seleccionados
  client_2: 50/50 

In [11]:
# ── S7: Per-Client F1 + PCD + PF ─────────────────────────────────────────
STRATEGY = 'S7'
print(f"\n{'='*60}\n🚀 Ejecutando {STRATEGY}\n{'='*60}")
for ds_name in DATASETS:
    print(f"  ▶ {ds_name}...", end=' ', flush=True)
    res = run_experiment(ds_name, STRATEGY)
    all_results[(ds_name, STRATEGY)] = res
    print(f"F1={res['global_macro_f1']:.4f}")
print(f"✅ {STRATEGY} completada.")


🚀 Ejecutando S7
  ▶ car... 
🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 27 árboles entrenados
  client_1: 43 árboles entrenados
  client_2: 24 árboles entrenados
  TOTAL: 94 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S7)
  client_0: 27/27 árboles seleccionados
  client_1: 43/43 árboles seleccionados
  client_2: 24/24 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 94 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 27 locales + 67 globales externos (27 propios excluidos) = 94 árboles
  client_1: 43 locales + 51 globales externos (43 propios excluidos) = 94 árboles
  client_2: 24 locales + 70 globales externos (24 propios excluidos) = 94 árboles

F1=0.8661
  ▶ iris... 
🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 9 árboles entrenados
  client_1: 24 árboles entrenados
  client_2: 16 árboles entrenados
  TOTAL: 49 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S7)
  client_0: 9/9 árboles seleccionados
  client_1: 24/24 árboles seleccionados
  client_2: 16/16 árb

In [12]:
# ── PW: Progressive Windows ──────────────────────────────────────────────
STRATEGY = 'PW'
print(f"\n{'='*60}\n🚀 Ejecutando {STRATEGY}\n{'='*60}")
for ds_name in DATASETS:
    print(f"  ▶ {ds_name}...", end=' ', flush=True)
    res = run_experiment(ds_name, STRATEGY)
    all_results[(ds_name, STRATEGY)] = res
    print(f"F1={res['global_macro_f1']:.4f}")
print(f"✅ {STRATEGY} completada.")


🚀 Ejecutando PW
  ▶ car... 
🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 24 árboles entrenados
  client_1: 43 árboles entrenados
  client_2: 24 árboles entrenados
  TOTAL: 91 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: PW)
  client_0: 24/24 árboles seleccionados
  client_1: 43/43 árboles seleccionados
  client_2: 24/24 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 91 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 24 locales + 67 globales externos (24 propios excluidos) = 91 árboles
  client_1: 43 locales + 48 globales externos (43 propios excluidos) = 91 árboles
  client_2: 24 locales + 67 globales externos (24 propios excluidos) = 91 árboles

F1=0.9091
  ▶ iris... 
🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 16 árboles entrenados
  client_1: 24 árboles entrenados
  client_2: 16 árboles entrenados
  TOTAL: 56 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: PW)
  client_0: 16/16 árboles seleccionados
  client_1: 24/24 árboles seleccionados
  client_2: 16/16 

---
## 4. Tabla comparativa final

Se calcula el **promedio de Accuracy y F1 de los 3 clientes** para cada combinación dataset × estrategia.

In [13]:
# ── Resultados base de Nayma (Proactive Forest) ──────────────────────────
NAYMA_RESULTS = {
    'car':       {'accuracy': 0.976625780, 'f1': 0.945782},
    'iris':      {'accuracy': 0.956000000, 'f1': 0.954981},
    'letter':    {'accuracy': 0.965045493, 'f1': 0.965253},
    'nursery':   {'accuracy': 0.995910870, 'f1': 0.954848},
    'optdigits': {'accuracy': 0.983235639, 'f1': 0.982218},
    'sonar':     {'accuracy': 0.848298701, 'f1': 0.823483},
    'spambase':  {'accuracy': 0.953879759, 'f1': 0.952755},
    'vowel':     {'accuracy': 0.971919192, 'f1': 0.968468},
}

STRATEGIES = ['S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'PW']
DATASET_ORDER = ['car', 'iris', 'letter', 'nursery', 'optdigits', 'sonar', 'spambase', 'vowel']

In [14]:
# ── Construir tabla comparativa ──────────────────────────────────────────
rows = []

for ds in DATASET_ORDER:
    row = {'BD': ds.capitalize()}

    # Columna PF (Nayma)
    row['Accuracy_PF'] = round(NAYMA_RESULTS[ds]['accuracy'], 6)
    row['F1_PF'] = round(NAYMA_RESULTS[ds]['f1'], 6)

    # Columnas por estrategia
    for strat in STRATEGIES:
        key = (ds, strat)
        if key in all_results:
            res = all_results[key]
            # Promedio de los 3 clientes
            client_accs = [m['accuracy'] for m in res['client_metrics'].values()]
            client_f1s = [m['macro_f1'] for m in res['client_metrics'].values()]
            row[f'Accuracy_{strat}'] = round(np.mean(client_accs), 6)
            row[f'F1_{strat}'] = round(np.mean(client_f1s), 6)
        else:
            row[f'Accuracy_{strat}'] = None
            row[f'F1_{strat}'] = None

    rows.append(row)

df_comparison = pd.DataFrame(rows)

# Reordenar columnas: BD, PF, luego cada estrategia
col_order = ['BD', 'Accuracy_PF', 'F1_PF']
for strat in STRATEGIES:
    col_order += [f'Accuracy_{strat}', f'F1_{strat}']
df_comparison = df_comparison[col_order]

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', lambda x: f'{x:.6f}' if pd.notna(x) else '')

print(df_comparison.to_string(index=False))

       BD  Accuracy_PF    F1_PF  Accuracy_S1    F1_S1  Accuracy_S2    F1_S2  Accuracy_S3    F1_S3  Accuracy_S4    F1_S4  Accuracy_S5    F1_S5  Accuracy_S6    F1_S6  Accuracy_S7    F1_S7  Accuracy_PW    F1_PW
      Car     0.976626 0.945782     0.953757 0.857976     0.966281 0.892709     0.963391 0.880686     0.952794 0.860743     0.964355 0.885124     0.961464 0.884652     0.947013 0.840493     0.957611 0.864536
     Iris     0.956000 0.954981     0.922222 0.922139     0.855556 0.854135     0.922222 0.922139     0.911111 0.910944     0.866667 0.866500     0.922222 0.922139     0.933333 0.933333     0.844444 0.843032
   Letter     0.965045 0.965253     0.907917 0.907524     0.911000 0.910577     0.910833 0.910425     0.912750 0.912370     0.913083 0.912723     0.913250 0.913055     0.909750 0.909428     0.910083 0.909526
  Nursery     0.995911 0.954848     0.976595 0.951544     0.977623 0.954740     0.979295 0.957729     0.975952 0.951960     0.977238 0.954467     0.974666 0.952650     

In [15]:
# ── Guardar tabla como CSV y Excel ───────────────────────────────────────
out_dir = ROOT / 'results' / 'comparison_vs_nayma'
out_dir.mkdir(parents=True, exist_ok=True)

df_comparison.to_csv(out_dir / 'comparison_table.csv', index=False)
df_comparison.to_excel(out_dir / 'comparison_table.xlsx', index=False)
print(f"\n✅ Tabla guardada en: {out_dir}")


✅ Tabla guardada en: C:\Users\Adrián Rodríguez\Documents\! Study 📝\📊 KDD 🤖🧠\! Federated learning\federated_proactive_forest\results\comparison_vs_nayma


In [16]:
# ── Tabla de diferencias vs Nayma (estrategia - PF) ─────────────────────
diff_rows = []

for ds in DATASET_ORDER:
    row = {'BD': ds.capitalize()}
    for strat in STRATEGIES:
        key = (ds, strat)
        if key in all_results:
            res = all_results[key]
            client_accs = [m['accuracy'] for m in res['client_metrics'].values()]
            client_f1s = [m['macro_f1'] for m in res['client_metrics'].values()]
            avg_acc = np.mean(client_accs)
            avg_f1 = np.mean(client_f1s)
            row[f'ΔAcc_{strat}'] = round(avg_acc - NAYMA_RESULTS[ds]['accuracy'], 6)
            row[f'ΔF1_{strat}'] = round(avg_f1 - NAYMA_RESULTS[ds]['f1'], 6)
        else:
            row[f'ΔAcc_{strat}'] = None
            row[f'ΔF1_{strat}'] = None
    diff_rows.append(row)

diff_cols = ['BD']
for strat in STRATEGIES:
    diff_cols += [f'ΔAcc_{strat}', f'ΔF1_{strat}']

df_diff = pd.DataFrame(diff_rows)[diff_cols]
print(df_diff.to_string(index=False))

df_diff.to_csv(out_dir / 'comparison_diff_vs_PF.csv', index=False)
print(f"\n✅ Diferencias guardadas en: {out_dir / 'comparison_diff_vs_PF.csv'}")

       BD   ΔAcc_S1    ΔF1_S1   ΔAcc_S2    ΔF1_S2   ΔAcc_S3    ΔF1_S3   ΔAcc_S4    ΔF1_S4   ΔAcc_S5    ΔF1_S5   ΔAcc_S6    ΔF1_S6   ΔAcc_S7    ΔF1_S7   ΔAcc_PW    ΔF1_PW
      Car -0.022869 -0.087806 -0.010344 -0.053073 -0.013235 -0.065096 -0.023832 -0.085039 -0.012271 -0.060658 -0.015161 -0.061130 -0.029612 -0.105289 -0.019015 -0.081246
     Iris -0.033778 -0.032842 -0.100444 -0.100846 -0.033778 -0.032842 -0.044889 -0.044037 -0.089333 -0.088481 -0.033778 -0.032842 -0.022667 -0.021648 -0.111556 -0.111949
   Letter -0.057129 -0.057729 -0.054045 -0.054676 -0.054212 -0.054828 -0.052295 -0.052883 -0.051962 -0.052530 -0.051795 -0.052198 -0.055295 -0.055825 -0.054962 -0.055727
  Nursery -0.019316 -0.003304 -0.018287 -0.000108 -0.016616  0.002881 -0.019959 -0.002888 -0.018673 -0.000381 -0.021245 -0.002198 -0.020731 -0.005792 -0.022531 -0.018809
Optdigits -0.023271 -0.022532 -0.029202 -0.028407 -0.027127 -0.026346 -0.026533 -0.025791 -0.022382 -0.021994 -0.025347 -0.024763 -0.021195 -0.020685 